In [1]:
import itertools
import tempfile
from copy import deepcopy
from math import comb
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from fsspec.implementations.dirfs import DirFileSystem
from toop_engine_dc_solver.example_grids import complex_grid_battery_hvdc_svc_3w_trafo_data_folder
from toop_engine_dc_solver.jax.split_modf_main_grid import (
    compute_split_modf_main_grid_from_mask,
    main_grid_reachable_masks_parallel,
)
from toop_engine_dc_solver.postprocess.postprocess_powsybl import PowsyblRunner, count_connected_components
from toop_engine_dc_solver.preprocess.convert_to_jax import load_grid
from toop_engine_dc_solver.preprocess.network_data import extract_action_set
from toop_engine_grid_helpers.powsybl.loadflow_parameters import SINGLE_SLACK
from toop_engine_interfaces.folder_structure import PREPROCESSING_PATHS
from toop_engine_interfaces.loadflow_result_helpers_polars import extract_solver_matrices_polars
from toop_engine_interfaces.messages.preprocess.preprocess_commands import AreaSettings
from toop_engine_interfaces.nminus1_definition import Contingency, GridElement, MonitoredElement, Nminus1Definition

jax.config.update("jax_enable_x64", True)  # doppelte Genauigkeit, sonst stimmt der Vergleich mit powsybl nicht

In [9]:
# Zelle 2: KONFIGURATION. Nur hier muss etwas eingetragen werden.
#
# DATEN_ORDNER = None  -> das Beispielnetz "complex grid" (mit 3-Wicklungs-Trafos, HVDC, Batterie, SVC, PSTs) wird mit
#                         dem Importer und der Vorverarbeitung des Repos neu erzeugt (dauert ca. 20 s).
# DATEN_ORDNER = Pfad  -> ein vollständig vom Importer erzeugter Ordner (grid.xiidm, nminus1_definition.json, Masken,
#                         Asset-Topologie), z.B. von node_breaker_folder_powsybl aus toop_engine_dc_solver.example_grids.
#                         Er wird mit load_grid vorverarbeitet. Ältere Ordner ohne Asset-Topologie gehen nicht.
DATEN_ORDNER = "/home/vc8596/ToOp/packages/dc_solver_pkg/src/toop_engine_dc_solver/mastergrid_cgmes_Datein_20260302/20260302T0030Z_CGMES_001.zip"

# Es werden alle Ausfallmengen mit 1 bis MAX_AUSFAELLE Zweigen geprüft, die das Netz in mehr Inseln teilen.
MAX_AUSFAELLE = 2

# Abweichungen unterhalb dieser Schwelle (in MW) gelten als übereinstimmend
TOL = 1e-6

# Woher kommen die Knoteneinspeisungen für split MODF?
#   "aus_fluessen"  = aus den N-0-Flüssen des Runners (Knotenregel). Passt exakt zum einzelnen Slack der Referenz.
#   "network_data"  = die Einspeisungen der Vorverarbeitung (sie entstehen mit verteiltem Slack, siehe Zelle 5)
EINSPEISUNG = "aus_fluessen"

# Wie viele Ausfallmengen werden zusätzlich direkt in powsybl auf mehr connected_components geprüft (Zelle 6)?
MAX_POWSYBL_PRUEFUNG = 500

# Für Zelle 7 (Tabelle Zweig für Zweig): Liste von Zweig-IDs, oder None = Fall mit der größten Abweichung
AUSFALL_ANSEHEN = None

In [10]:
# Zelle 3: Netz laden. Es werden die Funktionen des Repos benutzt, hier wird nichts selbst aufgebaut:
#   - Importer (convert_file): wandelt 3-Wicklungs-Trafos in drei 2W-Trafos "-Leg1" bis "-Leg3" um
#   - load_grid / preprocess: erzeugt NetworkData (PTDF, Zweig- und Knotenindizes, Slack)
if DATEN_ORDNER is None:
    ordner = Path(tempfile.mkdtemp(prefix="modf_test_3_"))
    gebiet = AreaSettings(
        cutoff_voltage=1.0,
        control_area=["BE", "NL", "DE"],
        view_area=["BE", "NL", "DE"],
        nminus1_area=["BE", "NL", "DE"],
        dso_trafo_factors=None,
        dso_trafo_weight=1.0,
        border_line_factors=None,
        border_line_weight=1.0,
    )
    network_data = complex_grid_battery_hvdc_svc_3w_trafo_data_folder(
        ordner, linear_pst=np.array([True, False, False]), area_settings=gebiet
    )
else:
    ordner = Path(DATEN_ORDNER)
    _, _, network_data = load_grid(DirFileSystem(str(ordner)), pandapower=False)

nd = network_data
n_branch = len(nd.branch_ids)
n_node = len(nd.node_ids)
slack = jnp.asarray(nd.slack, dtype=jnp.int32)
pd.DataFrame(
    {
        "Zweige": [n_branch],
        "Knoten": [n_node],
        "davon 3W-Wicklungen": [sum("-Leg" in b for b in nd.branch_ids)],
        "Knotentypen": [pd.Series(nd.node_types).value_counts().to_dict()],
    }
)

2026-11-26 16:07:37 [info     ] Preprocessing stage load_grid_into_loadflow_solver_backend, load into Powsybl backend message=load into Powsybl backend preprocess_stage=load_grid_into_loadflow_solver_backend


FileNotFoundError: /home/vc8596/ToOp/packages/dc_solver_pkg/src/toop_engine_dc_solver/mastergrid_cgmes_Datein_20260302/20260302T0030Z_CGMES_001.zip/grid.xiidm

In [4]:
# Zelle 4: Alle Ausfallmengen finden, die das Netz in mehr Inseln teilen.
# Dafür dient main_grid_reachable_masks_parallel aus dem Repo (Erreichbarkeit vom Slack-Bus, alle Mengen auf einmal).
# Eine Menge teilt das Netz, wenn danach ein Knoten nicht mehr erreichbar ist, der im intakten Netz erreichbar war.
# (Zweite Sammelschienen und PST-Hilfsknoten sind auch im intakten Netz nicht erreichbar und zählen deshalb nicht.)
def erreichbar(ausfaelle: np.ndarray) -> np.ndarray:
    return np.asarray(
        main_grid_reachable_masks_parallel(
            from_node=jnp.asarray(nd.from_nodes),
            to_node=jnp.asarray(nd.to_nodes),
            outages=jnp.asarray(ausfaelle, dtype=jnp.int32),
            slack_bus=slack,
            n_bus=n_node,
        )
    )


intakt = erreichbar(np.full((1, 1), -1))[0]
anzahl_kombinationen = sum(comb(n_branch, k) for k in range(1, MAX_AUSFAELLE + 1))
ausfallmengen = []
for k in range(1, MAX_AUSFAELLE + 1):
    kombinationen = np.array(list(itertools.combinations(range(n_branch), k)), dtype=np.int32)
    for start in range(0, len(kombinationen), 2000):
        # Ausfallmengen unterschiedlicher Größe werden mit -1 auf die gleiche Länge aufgefüllt (-1 = kein Zweig)
        stueck = np.pad(kombinationen[start : start + 2000], ((0, 0), (0, MAX_AUSFAELLE - k)), constant_values=-1)
        teilt = (intakt & ~erreichbar(stueck)).any(axis=1)
        ausfallmengen.extend(stueck[teilt])
ausfallmengen = np.array(ausfallmengen)
ausfall_namen = [", ".join(nd.branch_ids[b] for b in menge if b >= 0) for menge in ausfallmengen]
print(f"{len(ausfallmengen)} von {anzahl_kombinationen} Ausfallmengen teilen das Netz in mehr Inseln")

377 von 1485 Ausfallmengen teilen das Netz in mehr Inseln


In [5]:
# Zelle 5: Vergleich. Referenz = DC-Lastfluss von powsybl mit geöffneten Zweigen (PowsyblRunner aus dem Repo).
#   Split MODF  = compute_split_modf_main_grid_from_mask (nur die Insel mit dem Slack-Bus, sonst NaN)
#   powsybl     = N-1-Flüsse des Runners, alle Ausfallmengen in einem Aufruf
# Beide rechnen mit einzelnem Slack (SINGLE_SLACK). Der Slack-Bus kommt aus dem Netz (slackTerminal) und bleibt
# deshalb auch nach dem Öffnen von Zweigen derselbe.
def ausfall_definition(mengen: np.ndarray) -> Nminus1Definition:
    """N-1-Definition des Repos: ein Contingency je Ausfallmenge, überwacht werden alle Zweige."""
    ueberwacht = [
        MonitoredElement(id=i, name=n, type=t, kind="branch")
        for i, n, t in zip(nd.branch_ids, nd.branch_names, nd.branch_types, strict=True)
    ]
    ausfaelle = [Contingency(id="BASECASE", elements=[])]
    for nummer, menge in enumerate(mengen):
        elemente = [
            GridElement(id=nd.branch_ids[b], name=nd.branch_names[b], type=nd.branch_types[b], kind="branch")
            for b in menge
            if b >= 0
        ]
        ausfaelle.append(Contingency(id=f"outage-{nummer}", elements=elemente))
    return Nminus1Definition(monitored_elements=ueberwacht, contingencies=ausfaelle, id_type="powsybl")


runner = PowsyblRunner(lf_params=SINGLE_SLACK)
runner.load_base_grid(ordner / PREPROCESSING_PATHS["grid_file_path_powsybl"])
runner.store_action_set(extract_action_set(nd))
definition = ausfall_definition(ausfallmengen)
runner.store_nminus1_definition(definition)
n_0_roh, n_1_roh, erfolg = extract_solver_matrices_polars(runner.run_dc_loadflow([], []), definition, 0)
n_0_flow, n_1_flow, erfolg = -n_0_roh, -n_1_roh, np.asarray(erfolg)  # Vorzeichen wie im Solver

# Einspeisungen je Knoten (PST-Hilfsknoten haben keine physische Einspeisung)
inzidenz = np.zeros((n_branch, n_node))
inzidenz[np.arange(n_branch), nd.from_nodes] = 1.0
inzidenz[np.arange(n_branch), nd.to_nodes] = -1.0
if EINSPEISUNG == "aus_fluessen":
    einspeisung = inzidenz.T @ n_0_flow
else:
    einspeisung = np.asarray(nd.nodal_injection[0]).copy()
    einspeisung[np.asarray(nd.node_types) == "PSTNode"] = 0.0

haupt_knoten = erreichbar(ausfallmengen)  # die Insel mit dem Slack-Bus je Ausfallmenge
split_fluss, split_ueberwacht, split_bilanz, _rang = compute_split_modf_main_grid_from_mask(
    ptdf=jnp.asarray(nd.ptdf),
    from_node=jnp.asarray(nd.from_nodes),
    to_node=jnp.asarray(nd.to_nodes),
    nodal_injections=jnp.asarray(einspeisung[None, :]),
    n_0_flow=jnp.asarray(n_0_flow[None, :]),
    outages=jnp.asarray(ausfallmengen, dtype=jnp.int32),
    branches_monitored=jnp.arange(n_branch, dtype=jnp.int32),
    main_bus=jnp.asarray(haupt_knoten),
    slack_bus=slack,
)
split_fluss = np.asarray(split_fluss[0])  # Zeitschritt 0; NaN außerhalb der behaltenen Insel
split_ueberwacht = np.asarray(split_ueberwacht)  # Zweige der behaltenen Insel

# Abweichung nur auf den Zweigen der behaltenen Insel
differenz = np.where(split_ueberwacht, np.abs(split_fluss - n_1_flow), 0.0)
groesste = np.where(np.isnan(differenz), np.inf, differenz).max(axis=1)
ergebnis = pd.DataFrame(
    {
        "outaged_branches": ausfall_namen,
        "n_outages": (ausfallmengen >= 0).sum(axis=1),
        "n_kept_branches": split_ueberwacht.sum(axis=1),
        "max_diff_mw": groesste,
        "matches": (groesste <= TOL) & erfolg,
        "powsybl_ok": erfolg,
        "discarded_imbalance_mw": np.abs(np.asarray(split_bilanz)[0]),
    }
)
print(
    f"Ausfallmengen: {len(ergebnis)} | powsybl ohne Ergebnis: {int((~erfolg).sum())} | "
    f"abweichend: {int((~ergebnis.matches & ergebnis.powsybl_ok).sum())} | "
    f"größte Abweichung: {ergebnis[ergebnis.powsybl_ok].max_diff_mw.max():.3e} MW"
)
ergebnis.sort_values("max_diff_mw", ascending=False).head(10)

Ausfallmengen: 377 | powsybl ohne Ergebnis: 0 | abweichend: 0 | größte Abweichung: 2.234e-11 MW


,outaged_branches,n_outages,n_kept_branches,max_diff_mw,matches,powsybl_ok,discarded_imbalance_mw
282,"3W-Leg3, NL_3W_2-Leg3",2,53,2.233946e-11,True,True,1.000000e+01
283,"3W-Leg3, Dangling_outbound + Dangling_ch_inbound",2,53,2.160050e-11,True,True,3.000000e+02
280,"3W-Leg3, NL_3W_1-Leg3",2,53,1.995204e-11,True,True,2.000000e+01
277,"3W-Leg3, NL_3W_1-Leg1",2,51,1.995204e-11,True,True,2.000000e+01
286,"2W_MV_LV, NL_3W_1-Leg2",2,53,1.949729e-11,True,True,1.136868e-13
288,"2W_MV_LV, NL_3W_2-Leg2",2,53,1.921308e-11,True,True,1.500000e+01
157,"L21, NL_3W_1-Leg2",2,53,1.892886e-11,True,True,1.136868e-13
150,"L20, NL_3W_1-Leg2",2,53,1.892886e-11,True,True,1.136868e-13
19,"L2, NL_3W_1-Leg1",2,51,1.807621e-11,True,True,2.000000e+01
279,"3W-Leg3, NL_3W_1-Leg2",2,53,1.807621e-11,True,True,1.136868e-13


In [6]:
# Zelle 6: Steigt die Zahl der connected_components in powsybl wirklich? (Vorgabe: nur solche Fälle zählen.)
# count_connected_components kommt aus dem Repo. Die Zweige werden auf einer Kopie des Netzes entfernt, weil sich
# Zweige in Node-Breaker-Netzen (z.B. 3W-Wicklungen) mit disconnect() nicht immer abschalten lassen.
basis_komponenten = count_connected_components(runner.net)
stichprobe = np.arange(len(ausfallmengen))
if len(stichprobe) > MAX_POWSYBL_PRUEFUNG:
    stichprobe = np.sort(np.random.default_rng(0).choice(stichprobe, MAX_POWSYBL_PRUEFUNG, replace=False))


def komponenten_nach_ausfall(menge: np.ndarray) -> int:
    kopie = deepcopy(runner.net)
    kopie.remove_elements([nd.branch_ids[b] for b in menge if b >= 0])
    return count_connected_components(kopie)


komponenten = np.array([komponenten_nach_ausfall(ausfallmengen[i]) for i in stichprobe])
mehr = komponenten > basis_komponenten
ergebnis["powsybl_mehr_komponenten"] = pd.Series(mehr, index=stichprobe).reindex(ergebnis.index)
geprueft = ergebnis.loc[stichprobe]
mit_anstieg = geprueft[geprueft.powsybl_mehr_komponenten & geprueft.powsybl_ok]
pd.DataFrame(
    {
        "geprüfte Ausfallmengen": [len(stichprobe)],
        "Komponenten im intakten Netz": [basis_komponenten],
        "mehr Komponenten in powsybl": [int(mehr.sum())],
        "ohne Anstieg": [int((~mehr).sum())],
        "davon abweichend (nur Fälle mit Anstieg)": [int((~mit_anstieg.matches).sum())],
        "größte Abweichung mit Anstieg [MW]": [mit_anstieg.max_diff_mw.max()],
    }
)

,geprüfte Ausfallmengen,Komponenten im intakten Netz,mehr Komponenten in powsybl,ohne Anstieg,davon abweichend (nur Fälle mit Anstieg),größte Abweichung mit Anstieg [MW]
0,377,2,322,55,0,2.233946e-11


In [7]:
# Mengen, bei denen powsybl keine zusätzliche Komponente zählt, obwohl die Erreichbarkeit im NetworkData-Graph
# einen Knoten verliert. Meist ist der verlorene Knoten leer (keine Einspeisung), dann ist die verworfene Bilanz 0.
ohne_anstieg = ergebnis.loc[stichprobe[~mehr]]
ohne_anstieg[["outaged_branches", "n_kept_branches", "discarded_imbalance_mw", "max_diff_mw"]].head(15)

,outaged_branches,n_kept_branches,discarded_imbalance_mw,max_diff_mw
2,NL_3W_1-Leg2,53,1.136868e-13,0.000000e+00
12,"L1, NL_3W_1-Leg2",53,1.136868e-13,1.097078e-11
21,"L2, NL_3W_1-Leg2",53,1.136868e-13,1.608669e-11
28,"L3, NL_3W_1-Leg2",53,1.136868e-13,9.833911e-12
35,"L4, NL_3W_1-Leg2",53,1.136868e-13,3.296918e-12
42,"L5, NL_3W_1-Leg2",53,1.136868e-13,8.441248e-12
49,"L6, NL_3W_1-Leg2",53,1.136868e-13,6.593837e-12
54,"L7, 2W_MV_HV_PST",52,1.136868e-13,5.343281e-12
57,"L7, NL_3W_1-Leg2",53,1.136868e-13,5.286438e-12
62,"L8, 2W_MV_HV_1",52,1.136868e-13,2.330580e-12


In [8]:
# Zelle 7: Tabelle Zweig für Zweig für EINE Ausfallmenge (AUSFALL_ANSEHEN in Zelle 2).
if AUSFALL_ANSEHEN is None:
    fall = int(ergebnis.max_diff_mw.where(ergebnis.powsybl_ok, -1).to_numpy().argmax())
else:
    fall = ausfall_namen.index(", ".join(AUSFALL_ANSEHEN))
print("Ausfallmenge:", ausfall_namen[fall], "| verworfene Bilanz:", round(ergebnis.discarded_imbalance_mw[fall], 2), "MW")
tabelle = pd.DataFrame(
    {
        "ausgefallen": np.isin(np.arange(n_branch), ausfallmengen[fall]),
        "in behaltener Insel": split_ueberwacht[fall],
        "vor Ausfall [MW]": n_0_flow,
        "Split MODF [MW]": split_fluss[fall],
        "powsybl [MW]": n_1_flow[fall],
    },
    index=nd.branch_ids,
)
tabelle["Differenz [MW]"] = tabelle["Split MODF [MW]"] - tabelle["powsybl [MW]"]
tabelle.round(3)

Ausfallmenge: 3W-Leg3, NL_3W_2-Leg3 | verworfene Bilanz: 10.0 MW


,ausgefallen,in behaltener Insel,vor Ausfall [MW],Split MODF [MW],powsybl [MW],Differenz [MW]
L1,False,True,-43.667,25.000,25.000,0.0
L2,False,True,-16.333,-85.000,-85.000,-0.0
L3,False,True,107.146,114.736,114.736,0.0
L4,False,True,89.879,92.841,92.841,-0.0
L5,False,True,83.908,133.274,133.274,0.0
L6,False,True,-27.307,-21.976,-21.976,0.0
L7,False,True,79.036,77.961,77.961,-0.0
L8,False,True,-78.178,-81.985,-81.985,-0.0
L9,False,True,-78.178,-81.985,-81.985,-0.0
L10,False,True,31.794,22.886,22.886,-0.0
